<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# Construir una Asistente de Matemáticas de Inteligencia Artificial con la Herramienta LangChain

En este laboratorio, aprenderás a crear un agente sencillo con LangChain, lo que permitirá a los agentes de IA realizar tareas específicas. Crearás un conjunto de herramientas matemáticas que permitirá a un agente de IA realizar operaciones aritméticas básicas mediante la interacción en lenguaje natural.

A través de este laboratorio, crearás un agente capaz de comprender y resolver consultas matemáticas como "sumar 25 y 15, y luego multiplicar por 2", descomponiendo operaciones complejas en pasos sencillos.

# Table of Contents
6. [Relationship between agent and LLM](#Relationship-between-agent-and-LLM)
7. [Key parameters of initialize_agent](#Key-parameters-of-initialize_agent)
8. [Orchestrating multiple tools with an agent: Mathematical toolkit](#Orchestrating-multiple-tools-with-an-agent:-Mathematical-toolkit)
   - 8.1 [Subtraction tool](#Subtraction-tool)
9. [Building the agent](#Building-the-agent)
   - 9.1 [Exploring LangChain's built-in tools](#Exploring-LangChain's-built-in-tools)
   - 9.2 [Popular built-in tools](#Popular-built-in-tools)
   - 9.3 [Example: Using the Wikipedia tool](#Example:-Using-the-Wikipedia-tool)
10. [Exercise: Create a power tool to calculate exponents](#Exercise:-Create-a-power-tool-to-calculate-exponents)
    - 10.1 [Objective](#Objective)
    - 10.2 [Step 1: Create the power tool](#Step-1:-Create-the-power-tool)
    - 10.3 [Step 2: Create an agent with the power tool](#Step-2:-Create-an-agent-with-the-power-tool)
    - 10.4 [Step 3: Test the agent](#Step-3:-Test-the-agent)
11. [Authors](#authors)


## 1. Objetivos

Tras completar este laboratorio, podrás:

- Explicar el concepto de herramientas en LangChain
- Crear herramientas personalizadas para tareas específicas
- Desarrollar un agente de IA que pueda usar varias herramientas
- Depurar y mejorar la funcionalidad de las herramientas
- Probar las implementaciones de las herramientas con diferentes entradas

## 2. Configuración


Para este laboratorio, utilizarás las siguientes bibliotecas:

- **`langchain`**: Para crear agentes y herramientas de IA.

- **`langchain.chat_models`**: Para acceder a modelos de lenguaje.

- **`langchain.agents`**: Para crear y gestionar agentes de IA.

### *2.1. Instalación de las Bibliotecas Necesarias*

Las siguientes bibliotecas necesarias no están preinstaladas en el su entorno. Debe ejecutar la siguiente celda para instalarlas. Este paso puede tardar varios minutos, tenga paciencia.

```bash
pip install -r requirements.txt
```

### *2.2. Importación de las Bibliotecas Necesarias*

Importa aquí todas las bibliotecas necesarias:

In [12]:
# from langchain.agents import AgentType
from langchain.agents import AgentState

# from langchain_ibm import ChatWatsonx
from langchain_openai import ChatOpenAI

import re

## 3. Carga del LLM: Elección del Modelo de Lenguaje Adecuado


En este ejemplo, se utilizará `ChatWatsonx` de IBM para cargar un modelo de lenguaje (LLM) que permita interactuar con herramientas. Los modelos de IBM, como Granite 3.2 y Granite 3.3, son muy versátiles y destacan en tareas de razonamiento avanzado.

Sin embargo, otros proveedores ofrecen LLM con diferentes ventajas:

- **OpenAI (GPT-4/GPT-3.5)**: Ideal para versatilidad y razonamiento avanzado.

- **Facebook (Meta, LLaMA)**: De acceso abierto y altamente personalizable para casos de uso especializados.

- **IBM Watsonx Granite**: Perfecto para aplicaciones empresariales con una integración impecable.

- **Anthropic (Claude)**: Centrado en la seguridad, la fiabilidad y la IA ética.

- **Cohere**: Asequible y eficiente para modelos ligeros y específicos para tareas.


Para este proyecto, utilizarás `ChatWatsonx` porque:
- Ofrece una API sencilla para una configuración rápida.

- Admite configuraciones avanzadas como:

- **`temperatura`**: Ajusta la aleatoriedad de las respuestas.

- **`máx. tokens`**: Limita la longitud de las respuestas.

- Los modelos de IBM son ampliamente reconocidos como de vanguardia para el razonamiento y la conversación de propósito general.

In [16]:
# llm = ChatWatsonx(
#     apikey= "skills-network.eyJleHAiOjB9Cg==",
#     model_id= "ibm/granite-4-h-small",
#     project_id= "skills-network",    
#     url= "https://us-south.ml.cloud.ibm.com"
# )

llm = ChatOpenAI(
    model= 'gpt-4o-mini',
    max_completion_tokens= 512
)

Let's generate a simple response:


In [17]:
respuesta = llm.invoke("¿Qué es la llamada a herramientas en langchain?")
print("\nContenido de la respuesta: ", respuesta.content)


Contenido de la respuesta:  En Langchain, la "llamada a herramientas" se refiere a la capacidad de integrar y utilizar herramientas externas o APIs dentro de una aplicación construida con Langchain. Esto permite que los modelos de lenguaje, como los que se basan en GPT, puedan realizar tareas adicionales que no son posibles solo con su conocimiento interno. 

Por ejemplo, si un modelo necesita realizar búsquedas en internet, acceder a una base de datos, o utilizar funciones específicas como calcular información o generar gráficos, se puede hacer a través de llamadas a herramientas. Estas herramientas pueden ser APIs externas, servicios de terceros, o incluso funciones internas definidas por el desarrollador.

En un flujo de trabajo típico utilizando Langchain, el modelo de lenguaje puede recibir una consulta del usuario, decidir si necesita llamar a una herramienta para obtener más información y luego ejecutar esa llamada, procesar la respuesta y finalmente proporcionar una respuesta 

### *3.1. Aviso Legal Sobre la API*

Este laboratorio utiliza módulos de lenguaje natural (LLM) proporcionados por **IBM watsonx.ai** y **OpenAI**. Este entorno se ha configurado para permitir el uso de LLM sin claves API, de modo que pueda solicitarlos **gratis (con limitaciones)**. Por lo tanto, si desea ejecutar este cuaderno **localmente fuera** del entorno JupyterLab de Skills Network, deberá **configurar sus propias claves API**. Tenga en cuenta que el uso de sus propias claves API implicará cargos personales.

Si ejecuta este laboratorio localmente, deberá configurar sus propias claves API. Este laboratorio utiliza los módulos `ChatOpenAI` y `ChatWatsonx` de `langchain`. Ambas configuraciones se muestran a continuación con sus instrucciones. **Reemplace todas las instancias** de ambos módulos con los módulos completos que se muestran a continuación en todo el laboratorio. **NO** ejecute la celda que se muestra a continuación si no está trabajando localmente, ya que provocará errores.

## 4. Función

En IA, una **herramienta** invoca una **función** o capacidad básica que permite realizar una tarea específica. Imagínelo como una herramienta más en una caja de herramientas: al igual que un martillo, un destornillador o una llave inglesa, una caja de herramientas de IA contiene funciones específicas diseñadas para resolver problemas o realizar tareas.

Al crear herramientas para invocar otras herramientas, es importante tener en cuenta algunos principios clave:

1. **Propósito claro**: Asegúrese de que la herramienta tenga una función bien definida.

2. **Entrada estandarizada**: La herramienta debe aceptar la entrada en un formato predecible y estructurado para facilitar su uso.

3. **Resultados consistentes**: Siempre debe devolver resultados en un formato fácil de procesar o integrar con otros sistemas.

4. **Documentación completa**: Su herramienta debe incluir documentación clara y sencilla que explique su funcionamiento, cómo usarla y cualquier particularidad o limitación.

Recuerda que la documentación no es solo para otros desarrolladores, sino también para que el modelo de lenguaje (LLM) comprenda el propósito de la herramienta y cómo usarla eficazmente.

En este ejemplo, comenzaremos con una herramienta sencilla para sumar números. Cumplirá con la mayoría de los requisitos básicos, pero una limitación clave es que no maneja **errores básicos**, como ignorar entradas no numéricas. Mejorar el manejo de errores hará que la herramienta sea mucho más robusta y esté lista para su uso en situaciones reales.

In [19]:
def agregar_numeros(entradas: str) -> dict:
    """
    Agrega una lista de números proporcionada en un diccionario de entrada o extrae números de una cadena.
    Parámetros:
    - inputs (str):
    cadena que debe contener los números que se pueden extraer y sumar.

    Devuelve:
    - dict: Un diccionario con una sola clave "resultado" que contiene la suma de los números.

    Ejemplo de entrada (diccionario):
    {"numbers": [10, 20, 30]}

    Ejemplo de entrada (cadena):
    "Suma los números 10, 20 y 30."

    Ejemplo de salida:
    {"resultado": 60}
    """

    numbers = [int(x) for x in entradas.replace(",", "").split() if x.isdigit()]

    resultado = sum(numbers)

    return {"resultado": resultado}

Probar directamente una herramienta permite identificar dónde reside el problema, ya sea en su lógica, en el análisis de la entrada o en el formato de la salida.

Aquí, introducirás la cadena `1 2` y obtendrás la suma.

In [20]:
agregar_numeros("1 2") 

{'resultado': 3}

### *4.1. Herramienta*

La clase `Tool` en LangChain actúa como un contenedor estructurado que convierte funciones Python estándar en herramientas compatibles con agentes. Cada herramienta requiere tres componentes clave:
1. Un nombre que la identifique
2. La función que realiza la operación
3. Una descripción que ayude al agente a comprender cuándo usar la herramienta

Mejora de la sección de pruebas:

In [24]:
from langchain_core.tools import Tool

agregar_herramienta = Tool(
        name= "AgregarHerramienta",
        func= agregar_numeros,
        description= "Agregar una lista de números y devolver la suma de ellos.")

print("Objeto de herramienta:", agregar_herramienta)

Objeto de herramienta: name='AgregarHerramienta' description='Agregar una lista de números y devolver la suma de ellos.' func=<function agregar_numeros at 0x7d6f9e543a60>


Let's see the parameters of the object:

- **`name`** (*str*):
  - A unique identifier for the tool.

  - **Example**: `"AddTool"`

- **`.invoke`** (*Callable*):
  - The function that the tool wraps.
  - **Example**: `add_numbers`

- **`description`** (*str*):
  - A concise explanation of what the tool does.
  - **Example**: `"Adds a list of numbers and returns the result."`


Veamos los parámetros del objeto:

- **`name`** (*str*):
  - Un identificador único para la herramienta.

  - **Ejemplo**: `"AgregarHerramienta"`

- **`.invoke`** (*Callable*):
  - La función que encapsula la herramienta.

  - **Ejemplo**: `agregar_numeros`

- **`description`** (*str*):
  - Una breve explicación de la función de la herramienta.
  
  - **Ejemplo**: `"Suma una lista de números y devuelve el resultado."`

Estos atributos le permiten inspeccionar el objeto de la herramienta.

In [ ]:
# Nombre de la herramienta.
print("Nombre de la herramienta:")
print(agregar_herramienta.name)

# Descripción de la herramienta.
print("Descripción de la herramienta:")
print(agregar_herramienta.description)

# Función de la herramienta
print("Función de la herramienta:")
print(agregar_herramienta.invoke)


Nombre de la herramienta:
AgregarHerramienta
Descripción de la herramienta:
Agregar una lista de números y devolver la suma de ellos.
Función de la herramienta:
<bound method BaseTool.invoke of Tool(name='AgregarHerramienta', description='Agregar una lista de números y devolver la suma de ellos.', func=<function agregar_numeros at 0x7d6f9e543a60>)>


Puedes llamar a la función de la herramienta a través del objeto ```agregar_herramienta```:

In [29]:
print("Llamando a la función de la herramienta:")

test_input = "10 20 30 a b" 

print(agregar_herramienta.invoke(test_input))  # Ejemlo.

Llamando a la función de la herramienta:
{'resultado': 60}


Las pruebas del objeto de la herramienta garantizan lo siguiente:

1. **La herramienta está configurada correctamente**:
   - Los metadatos (nombre, descripción, etc.) están definidos correctamente y se ajustan a su propósito.

   - La función y el esquema (si corresponde) están configurados correctamente.

2. **La función encapsulada se comporta como se espera**:
   - La función realiza la tarea prevista correctamente.

   - Gestiona adecuadamente los casos límite y las entradas no válidas.

3. **La herramienta se integra sin problemas con los agentes**:
   - La salida de la herramienta coincide con lo que espera el agente.

   - No hay problemas de compatibilidad cuando el agente llama a la herramienta.


### 4.2. Operador `@tool`

Ahora que ya sabes cómo crear una herramienta con la clase `Tool` (usando la interfaz Tool), existe otra forma de crearla: mediante el decorador `@tool`. El método recomendado para crear herramientas es usar el decorador `@tool`. Este decorador está diseñado para simplificar el proceso de creación de herramientas y debería usarse en la mayoría de los casos. Tras definir una función, puedes decorarla con `@tool` para crear una herramienta que implemente la interfaz Tool.

El operador `@tool` convierte funciones en herramientas. Ver más abajo:

In [30]:
from langchain_core.tools import tool
import re

@tool
def agregar_numeros(entradas: str) -> dict:
    """
    Agrega una lista de números proporcionada en un diccionario de entrada o extrae números de una cadena.
    Parámetros:
    - inputs (str):
    cadena que debe contener los números que se pueden extraer y sumar.

    Devuelve:
    - dict: Un diccionario con una sola clave "resultado" que contiene la suma de los números.

    Ejemplo de entrada (diccionario):
    {"numbers": [10, 20, 30]}

    Ejemplo de entrada (cadena):
    "Suma los números 10, 20 y 30."

    Ejemplo de salida:
    {"resultado": 60}
    """

    numbers = [int(x) for x in entradas.replace(",", "").split() if x.isdigit()]

    resultado = sum(numbers)

    return {"resultado": resultado}

La función anterior ahora actuará como una herramienta. Puede inspeccionar el esquema de la herramienta y otras propiedades utilizando:

In [31]:
print("Nombre: \n", agregar_numeros.name)
print("Descripción: \n", agregar_numeros.description) 
print("Args: \n", agregar_numeros.args) 


Nombre: 
 agregar_numeros
Descripción: 
 Agrega una lista de números proporcionada en un diccionario de entrada o extrae números de una cadena.
Parámetros:
- inputs (str):
cadena que debe contener los números que se pueden extraer y sumar.

Devuelve:
- dict: Un diccionario con una sola clave "resultado" que contiene la suma de los números.

Ejemplo de entrada (diccionario):
{"numbers": [10, 20, 30]}

Ejemplo de entrada (cadena):
"Suma los números 10, 20 y 30."

Ejemplo de salida:
{"resultado": 60}
Args: 
 {'entradas': {'title': 'Entradas', 'type': 'string'}}


Puedes llamar a la herramienta usando el método ```invoke```.

In [33]:
test_input = "Cual es la suma de 10, 20 y 30 " 

print(agregar_numeros.invoke(test_input))  # Ejemplo.

{'resultado': 60}


### *4.3. Usar @tool-StructuredTool*

El decorador @tool crea una StructuredTool con información de esquema extraída de las firmas de las funciones y las cadenas de documentación, como se muestra aquí. Esto ayuda a los desarrolladores de lenguajes de programación a comprender mejor qué entradas espera la herramienta y cómo usarla correctamente. Si bien ambos enfoques funcionan, @tool suele ser la opción preferida para las aplicaciones modernas de LangChain, especialmente con LangGraph y modelos de llamada a funciones.


In [36]:
# Comparación de los dos enfoques.
print("Enfoque de construcción de herramientas:")

print(f"Tiene esquema: {hasattr(agregar_herramienta, 'args_schema')}")
print("\n")

print("@tool Enfoque de decoración:")


print(f"Tiene esquema: {hasattr(agregar_numeros, 'args_schema')}")
print(f"Argumentos: {agregar_numeros.args}")

Enfoque de construcción de herramientas:
Tiene esquema: True


@tool Enfoque de decoración:
Tiene esquema: True
Argumentos: {'entradas': {'title': 'Entradas', 'type': 'string'}}


En este ejemplo, la herramienta tiene dos entradas: una cadena de texto que contiene los números a sumar y una segunda entrada booleana que determina si se deben sumar los valores absolutos de esos números.

In [37]:
from typing import List

@tool
def agregar_numeros_con_opcion(numeros: List[float], absoluto: bool = False) -> float:
    """
    Agrega una lista de números proporcionada como entrada.

    Parámetros:
    - numeros (List[float]): Una lista de números que se sumarán.
    - absoluto (bool): Si es True, utiliza los valores absolutos de los números antes de sumar.

    Devuelve:
    - float: La suma total de los números.
    """
    if absoluto:
        numeros = [abs(n) for n in numeros]

    return sum(numeros)

Comparemos los argumentos de agregar_numeros_con_opcion y agregar_numeros. Ambas son herramientas estructuradas. Ambas incluyen el campo inputs, que es una cadena de texto. Sin embargo, agregar_numeros_con_opcion tiene un par clave-valor adicional: absoluto, un campo booleano con un valor predeterminado de False. Esto significa que agregar_numeros_con_opcion admite un comportamiento opcional (tomar el valor absoluto de los números), mientras que agregar_numeros solo maneja la extracción y suma numérica básica.

In [40]:
print(f"Argumentos: {agregar_numeros.args}")
print(f"Argumentos: {agregar_numeros_con_opcion.args}")

Argumentos: {'entradas': {'title': 'Entradas', 'type': 'string'}}
Argumentos: {'numeros': {'items': {'type': 'number'}, 'title': 'Numeros', 'type': 'array'}, 'absoluto': {'default': False, 'title': 'Absoluto', 'type': 'boolean'}}


Puedes llamar a la herramienta usando un diccionario como entrada, donde cada clave corresponde a un nombre de parámetro y el valor es la entrada para ese parámetro. Por ejemplo, para controlar si los números se suman normalmente o con valores absolutos, establece el indicador absoluto en Falso o Verdadero: Obtendrás -6 y 6, respectivamente.

In [42]:
print(agregar_numeros_con_opcion.invoke({"numeros": [-1.1, -2.1, -3.0], "absoluto": False}))
print(agregar_numeros_con_opcion.invoke({"numeros": [-1.1, -2.1, -3.0], "absoluto": True}))

-6.2
6.2


### *4.4. Tipos de Retorno de Herramientas Mejorados con Tipado Python*

Al crear herramientas, debe especificar con precisión sus valores de retorno. Esto ayuda al agente a comprender y gestionar las diferentes salidas posibles.

La función `sumar_numeros_con_salida_compleja` devuelve un formato de salida más flexible. Devuelve un diccionario que contiene un valor flotante cuando los números se suman correctamente, o un mensaje de error descriptivo en formato de cadena si no se encuentran números o si se produce algún problema durante el procesamiento.

In [43]:
from typing import Dict, Union

@tool
def sumar_numeros_con_salida_compleja(entradas: str) -> Dict[str, Union[float, str]]:
    """
    Extrae y suma todos los números enteros y decimales de la cadena de entrada.

    Parámetros:
    - entradas (str): Una cadena que puede contener valores numéricos.

    Devuelve:
    - dict: Un diccionario con la clave "resultado". Si se encuentran números, el valor es su suma (float).
            Si no se encuentran números o se produce un error, el valor es un mensaje correspondiente (str).

    Ejemplo de entrada:
    "Suma 10, 20.5 y -3."

    Ejemplo de salida:
    {"resultado": 27.5}
    """

    matches = re.findall(r'-?\d+(?:\.\d+)?', entradas)

    if not matches:
        return {"resultado": "No se encontraron números en la entrada."}
    try:
        numeros = [float(num) for num in matches]
        total = sum(numeros)

        return {"resultado": total}
    
    except Exception as e:        
        return {"resultado": f"Error durante la suma: {str(e)}"}

La función `sumar_numeros_desde_texto` devuelve un formato de salida sencillo. Extrae todos los valores enteros de la cadena de entrada, los suma y devuelve el total como un número decimal. Esta función presupone que hay al menos un número válido en la entrada y no gestiona los casos en los que no se encuentran números o en los que puede producirse un error.

In [44]:
@tool
def sumar_numeros_desde_texto(entradas: str) -> float:
    """    
    Agrega una lista de números proporcionada en la cadena de entrada.

    Argumentos:
        entradas: Una cadena que contiene los números que se extraerán y sumarán.

    Devuelve:
        La suma de todos los números encontrados en la entrada.
    """
    # Usar expresiones regulares para extraer números enteros de la cadena de entrada.
    numeros = [int(num) for num in re.findall(r'\d+', entradas)]
    resultado = sum(numeros)

    return resultado

### 4.5. `initialize_agent`

Al configurar un agente, se conectan herramientas y un LLM para que trabajen juntos sin problemas. El agente utiliza el LLM para comprender qué se necesita hacer y decide qué herramienta usar según la tarea. Aquí hay una descripción general de los componentes clave:

**Relación Entre el Agente y el LLM**
- El **agente** actúa como el responsable de la toma de decisiones, determinando qué herramientas usar y cuándo.

- El **LLM** es el motor de razonamiento. Este:

    - Interpreta la entrada del usuario.

    - Ayuda al agente a tomar decisiones.

    - Genera una respuesta basada en la salida de las herramientas.

Imagina al agente como el administrador que asigna tareas y al LLM como el cerebro que resuelve problemas o delega trabajo.

**Parámetros clave de `initialize_agent`**

1. **`tools`** - ver arriba

2. **`llm`** - ver arriba

3. **`agent`**:
  - Especifica el marco de razonamiento del agente.

  - `"zero-shot-react-description"` habilita:
    - **Razonamiento de cero disparos**: El agente puede resolver tareas que no ha visto antes analizando el problema paso a paso.

    - **Marco de React**: Un bucle lógico de:

      - **Razonar** → Pensar en la tarea.

      - **Actuar** → Usar una herramienta para realizar una acción.

      - **Observar** → Comprobar la salida de la herramienta.

      - **Planificar** → Decidir qué hacer a continuación.

4. **`verbose`**:
- Si es `True`, imprime registros detallados del proceso de pensamiento del agente.

- Útil para depurar o comprender cómo el agente toma decisiones.

You can now create an agent object using initialize_agent.


In [57]:
from langchain.agents import create_agent

agente = create_agent(model= 'gpt-4o-mini', tools= [agregar_herramienta])

Ahora, puedes poner a prueba al agente haciéndole una pregunta.

> Al ejecutar un agente mediante `.run()` o `.invoke()`, es posible que ocasionalmente se encuentre con una situación en la que el código se ejecute indefinidamente, incluso si el LLM ya ha generado una respuesta válida. Esto suele ocurrir cuando el sistema encuentra un `OUTPUT_PARSING_ERROR`, a menudo debido a problemas de formato en la respuesta del LLM.
>
>En estos casos, el agente puede quedar bloqueado en un bucle y no finalizará por sí solo. Si observa que esto sucede, simplemente haga clic en el botón de detener (■) en la barra de herramientas superior para interrumpir la ejecución.

In [61]:
# Usar el agente para realizar una tarea que requiere la herramienta de suma.

respuesta = agente.invoke({'messages': "En 2023, el PIB de EE. UU. fue de aproximadamente 27,72 billones de dólares, mientras que el de Canadá fue de alrededor de 2,14 billones de dólares y el de México de aproximadamente 1,79 billones de dólares. ¿Cuál es el total?"})

In [62]:
respuesta

{'messages': [HumanMessage(content='En 2023, el PIB de EE. UU. fue de aproximadamente 27,72 billones de dólares, mientras que el de Canadá fue de alrededor de 2,14 billones de dólares y el de México de aproximadamente 1,79 billones de dólares. ¿Cuál es el total?', additional_kwargs={}, response_metadata={}, id='44c56d95-c4a0-4d7b-91dc-ac592f1a4404'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 122, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_d0a1738203', 'id': 'chatcmpl-DZrPhEiGbDjJy3tsdyY2VXiIiYtd2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd7a1-60b7-70a2-83e3-b4df2377b

In [65]:
agente.invoke({"messages": "Suma 10, 20, dos y 30"})

{'messages': [HumanMessage(content='Suma 10, 20, dos y 30', additional_kwargs={}, response_metadata={}, id='40fbdfd3-a6d7-4bfa-b5a5-93113852738e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 72, 'total_tokens': 101, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8a57a0807f', 'id': 'chatcmpl-DZrRUxNWLcGSlSP19M7xm2ZEJNvWh', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd7a3-1976-7271-a383-a305f2db442a-0', tool_calls=[{'name': 'AgregarHerramienta', 'args': {'__arg1': '10, 20, 2, 30'}, 'id': 'call_YjIxDt27nV50xK7PP66tmUJV', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 7

Se le pidió al agente que sumara los números 10, 20, "dos" y 30. El agente notó primero que una de las entradas era la palabra "dos" en lugar de un número, por lo que la convirtió a su forma numérica, que es 2. Después de preparar la lista de números (10, 20, 2 y 30), el agente decidió usar la herramienta `AddTool` para realizar la suma. Le pasó los números a la herramienta, que calculó la suma y devolvió el resultado como 62. Finalmente, el agente proporcionó la respuesta: **62**.

#### **Structured chat zero shot react-description**

When selecting an agent in LangChain, two factors matter: the agent type and the tool format, especially the tool’s return type. Agents like zero-shot-react-description expect tools to take and return plain strings, which works well with manually defined Tool(...) wrappers. 

In contrast, structured agents like `structured-chat-zero-shot-react-description` or `openai-functions` are built to handle structured inputs and outputs via the @tool decorator. If a tool returns a dictionary but the agent expects a string, it can cause key errors or parsing failures. 


In the agent example below, use `sum_numbers_from_text` as a tool and `structured-chat-zero-shot-react-description` as the agent type. For the LLM, you'll use `Granite`.


In [ ]:
agent_2 = initialize_agent([sum_numbers_from_text], llm, agent="structured-chat-zero-shot-react-description", verbose=True, handle_parsing_errors=True)
response = agent_2.invoke({"input": "Add 10, 20 and 30"})
print(response)

Now, for the below agent, you will be using `sum_numbers_with_complex_output` as the tool. As for the LLM, you are going to use `gpt-4.1-nano` and the agent type `openai-functions`. 

One thing to note here is Some LLMs, like `Granite`, cannot unpack dictionary outputs because they lack native support for structured output parsing. As a result, when you use `sum_numbers_with_complex_output` with the `structured-chat-zero-shot-react-description` agent type, the agent fails to interpret the returned dictionary and throws an input validation or parsing error.


In [ ]:
from langchain_openai import ChatOpenAI

llm_ai = ChatOpenAI(model="gpt-4.1-nano")

In [ ]:
agent_3 = initialize_agent([sum_numbers_with_complex_output], llm_ai, agent="openai-functions", verbose=True, handle_parsing_errors=True)
response = agent_3.invoke({"input": "Add 10, 20 and 30"})
print(response)

Now, let's move on to tools with multiple inputs. The agent below uses `Granite` as the LLM and `add_numbers_with_options` as the tool, which accepts multiple input parameters. However, if the tool returns a complex output—such as a dictionary like in `sum_numbers_with_complex_output`—you’ll need to switch to a model like GPT and use an agent type that supports both multi-input tools and structured outputs. Granite and similar models may not handle complex output parsing reliably, especially when used with agents like `structured-chat-zero-shot-react-description`.


In [ ]:
agent_2 = initialize_agent(
    [add_numbers_with_options],
    llm,
    agent="structured-chat-zero-shot-react-description",
    verbose=True
)

In [ ]:
response = agent_2.invoke({
    "input": "Add -10, -20, and -30 using absolute values."
})
print(response)

Let's try with OpenAI to see if it runs with multiple inputs.


In [ ]:
agent_openai = initialize_agent(
    [add_numbers_with_options],
    llm_ai,
    agent="openai-functions",
    verbose=True
)

In [ ]:
response = agent_openai.invoke({
    "input": "Add -10, -20, and -30 using absolute values."
})
print(response)

### **`create_react_agent`**

As LangChain's `AgentExecutor` is being deprecated, create_react_agent from LangGraph provides a more flexible and powerful alternative for building AI agents. This function creates a graph-based agent that works with chat models and supports tool-calling functionality.

---

#### **Key parameters of `create_react_agent`**

1. **`model`**
    - The language model that powers the agent's reasoning.
    - Must support tool calling for full functionality.

2.  **`tools`**
    - A list of tools the agent can use to perform actions.
    - Can be LangChain tools, Python functions with @tool decorator, or a ToolNode instance
    - Each tool should have a name, description, and implementation

3. **`prompt (optional)`**:
   - Customizes the instructions given to the LLM
   - Can be:
        - A string (converted to a SystemMessage)
        - A SystemMessage object
        - A function that transforms the state
        - A Runnable that processes the state

and other parameters. To see more parameters, see [docs](https://langchain-ai.github.io/langgraph/reference/prebuilt/).

#### How it works

Unlike the legacy `AgentExecutor`, which used a fixed loop structure, `create_react_agent` creates a graph with these key nodes:

1. **Agent Node:** Calls the LLM with the message history
2. **Tools Node:** Executes any tool calls from the LLM's response
3. **Continue/End Nodes:** Manage the workflow based on whether tool calls are present

The graph follows this process:

1. User message enters the graph
2. LLM generates a response, potentially with tool calls
3. If tool calls exist, they're executed and their results are added to the message history
4. The updated messages are sent back to the LLM
5. This loop continues until the LLM responds without tool calls
6. The final state with all messages is returned


In [ ]:
%pip install langgraph==0.6.1 | tail -n 1

In [ ]:
from langgraph.prebuilt import create_react_agent

agent_exec = create_react_agent(model=llm_ai, tools=[sum_numbers_from_text])
msgs = agent_exec.invoke({"messages": [("human", "Add the numbers -10, -20, -30")]})

In [ ]:
print(msgs["messages"][-1].content)

## Orchestrating multiple tools with an agent: Mathematical toolkit
In real-world applications, a single tool is often insufficient to address the complexity and diversity of user requests. Tasks such as data analysis, performing calculations, or retrieving specific information require specialized capabilities that cannot be fulfilled by a single function. By equipping an agent with multiple tools, each tailored to a distinct purpose, you'll create a system that can dynamically select and utilize the appropriate tool based on the user’s query. This approach enhances the flexibility and scalability of the AI, allowing it to handle a broad spectrum of tasks with precision and efficiency. The orchestration of multiple tools ensures that the agent can seamlessly manage complex workflows, making it an essential framework for building robust and versatile AI systems.

To demonstrate this concept, let’s create additional tools, i.e, a mathematical toolkit. In addition to the addition tool, you will now create tools for subtraction, multiplication, and division. These tools will be integrated into an agent capable of handling various mathematical queries, showcasing how multiple tools can work together within a single AI system.

### Subtraction tool
The subtraction tool is designed to take a list of numbers and return the result of subtracting all subsequent numbers from the first number. This tool is particularly useful for handling queries involving differences, such as "What is 100 minus 20 and then minus 10?". 


In [ ]:
@tool
def subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string, negates the first number, and successively subtracts 
    the remaining numbers in the list.

    This function is designed to handle input in string format, where numbers are separated 
    by spaces, commas, or other delimiters. It parses the string, extracts valid numeric values, 
    and performs a step-by-step subtraction operation starting with the first number negated.

    Parameters:
    - inputs (str): 
      A string containing numbers to subtract. The string may include spaces, commas, or 
      other delimiters between the numbers.

    Returns:
    - dict: 
      A dictionary containing the key "result" with the calculated difference as its value. 
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Input:
    "100, 20, 10"

    Example Output:
    {"result": -130}

    Notes:
    - Non-numeric characters in the input are ignored.
    - If the input string contains only one valid number, the result will be that number negated.
    - Handles a variety of delimiters (e.g., spaces, commas) but does not validate input formats 
      beyond extracting numeric values.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]

    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Start with the first number negated
    result = -1 * numbers[0]

    # Subtract all subsequent numbers
    for num in numbers[1:]:
        result -= num

    return {"result": result}

You can inspect the tool's schema and other properties using:


In [ ]:
print("Name: \n", subtract_numbers.name)
print("Description: \n", subtract_numbers.description) 
print("Args: \n", subtract_numbers.args) 

Let's use it directly by calling the function:


In [ ]:
print("Calling Tool Function:")
test_input = "10 20 30 and four a b" 
print(subtract_numbers.invoke(test_input))  # Example

Let's now build multiple tools, starting with `MultiplyTool` and `DivideTool`, by defining two functions: `multiply_numbers` and `divide_numbers`. These functions are simple - `multiply_numbers` takes a list of numbers in string format and returns their product, while `divide_numbers` takes the first number and divides it by each subsequent number in sequence. Instead of manually wrapping these functions in the Tool class, you'll use the `@tool` decorator to automatically convert them into LangChain tools, using their docstrings as descriptions. These decorated tools can then be added directly to the agent alongside other operations like addition or subtraction, allowing the agent to intelligently select the appropriate operation based on the user's query, making it versatile for handling various math problems.


In [ ]:
# Multiplication Tool
@tool
def multiply_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates their product.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the product of the numbers.

    Example Input:
    "2, 3, 4"

    Example Output:
    {"result": 24}

    Notes:
    - If no numbers are found, the result defaults to 1 (neutral element for multiplication).
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]
    print(numbers)

    # If no numbers are found, return 1
    if not numbers:
        return {"result": 1}

    # Calculate the product of the numbers
    result = 1
    for num in numbers:
        result *= num
        print(num)

    return {"result": result}

In [ ]:
# Division Tool
@tool
def divide_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates the result of dividing the first number 
    by the subsequent numbers in sequence.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the quotient.

    Example Input:
    "100, 5, 2"

    Example Output:
    {"result": 10.0}

    Notes:
    - If no numbers are found, the result defaults to 0.
    - Division by zero will raise an error.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]


    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Calculate the result of dividing the first number by subsequent numbers
    result = numbers[0]
    for num in numbers[1:]:
        result /= num

    return {"result": result}

When testing these mathematical tools directly, notice that using raw string inputs like "2, 3, and four" or "100, 5, two" will fail. The tools are designed to work with numeric inputs only - they don't have the natural language understanding that comes with the LLM agent layer. To test properly, you need to use a numeric value:


In [ ]:
# Testing multiply_tool
multiply_test_input = "2, 3, and four "
multiply_result = multiply_numbers.invoke(multiply_test_input)
print("--- Testing MultiplyTool ---")
print(f"Input: {multiply_test_input}")
print(f"Output: {multiply_result}")

In [ ]:
# Testing divide_tool
divide_test_input = "100, 5, two"
divide_result = divide_numbers.invoke(divide_test_input)
print("--- Testing DivideTool ---")
print(f"Input: {divide_test_input}")
print(f"Output: {divide_result}")

## Building the agent

With the implementation of mathematical operators—addition, subtraction, multiplication, and division — you have established a simple yet functional mathematical toolkit. Unlike before, the agent must now not only select the appropriate tool and process the input but also determine the correct mathematical operation based on the user's query.

Let's create the agent object. first, combine all tools into a single list:


In [ ]:
tools = [add_numbers,subtract_numbers, multiply_numbers, divide_numbers]
tools

Like before, you will create the agent with the tools and the language model as input.


In [ ]:
from langgraph.prebuilt import create_react_agent

# Create the agent with all tools
math_agent = create_react_agent(
    model=llm_ai,
    tools=tools,
    # Optional: Add a system message to guide the agent's behavior
    prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly."
)

In [ ]:
response = math_agent.invoke({
    "messages": [("human", "What is 25 divided by 4?")]
})

# Get the final answer
final_answer = response["messages"][-1].content
print(final_answer)

In [ ]:
response_2 = math_agent.invoke({
    "messages": [("human", "Subtract 100, 20, and 10.")]
})

# Get the final answer
final_answer_2 = response_2["messages"][-2].content
print(final_answer_2)

When the agent tries to subtract 20 and 10 from 100, something unexpected happens. The tool called `SubtractTool` works differently than the agent expected. When you type in "100, 20, 10", instead of giving you 70 like you'd expect, it gives you -130. This happens because your special calculator first turns 100 into -100, then subtracts the other numbers.

```The Confusion``` 

The agent expects the function to work like normal math 100 - 20 - 10 = 70). When the agent tries to fix this by breaking the problem into smaller steps, it still gets unexpected answers because the calculator keeps using its special rules.


```Getting Stuck```

 The agent keeps trying the same approach repeatedly, not realizing why it isn't working. Eventually, it runs out of time without solving the problem.
 
 Before you fix the problem, let's test the other tools.


In [ ]:
print("\n--- Testing MultiplyTool ---")
response = math_agent.invoke({
    "messages": [("human", "Multiply 2, 3, and four.")]
})
print("Agent Response:", response["messages"][-1].content)

print("\n--- Testing DivideTool ---")
response = math_agent.invoke({
    "messages": [("human", "Divide 100 by 5 and then by 2.")]
})
print("Agent Response:", response["messages"][-1].content)

Now lets change the `SubtractTool` so it subtracts the numbers directly (without negating the first number). This aligns the tool’s behavior with standard arithmetic and the agent’s expectations.


In [ ]:
@tool
def new_subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and performs subtraction sequentially, starting with the first number.

    This function is designed to handle input in string format, where numbers may be separated by spaces, 
    commas, or other delimiters. It parses the input string, extracts numeric values, and calculates 
    the result by subtracting each subsequent number from the first. inputs[0]-inputs[1]-inputs[2]

    Parameters:
    - inputs (str): 
      A string containing numbers to subtract. The string can include spaces, commas, or other 
      delimiters between the numbers.

    Returns:
    - dict: 
      A dictionary containing the key "result" with the calculated difference as its value. 
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Usage:
    - Input: "100, 20, 10"
    - Output: {"result": 70}

    Limitations:
    - The function does not handle cases where numbers are formatted with decimals or other non-integer representations.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]

    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Start with the first number
    result = numbers[0]

    # Subtract all subsequent numbers
    for num in numbers[1:]:
        result -= num

    return {"result": result}


## Note: Tool naming when demonstrating different approaches

In this lab, two different ways to create the same mathematical tool (addition) are intentionally shown:

1. Using the `Tool()` constructor approach (`add_tool`)
2. Using the `@tool` decorator approach (`add_numbers`)

This is to compare different LangChain tool creation methods for educational purposes. In a production application, you would typically choose one consistent approach rather than having duplicate tools for the same functionality.

When building real agents, duplicate tools with similar functions will confuse the LLM, as it won't know which one to choose. Always use unique tools with clearly differentiated purposes in production code.


Next, create a new agent, ensuring it incorporates the updated subtraction tool.


In [ ]:
tools_updated = [add_numbers, new_subtract_numbers, multiply_numbers, divide_numbers]
# Create the agent with all tools
math_agent_new = create_react_agent(
    model=llm,
    tools=tools_updated,
    # Optional: Add a system message to guide the agent's behavior
    prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly."
)
print("agent",math_agent_new)

Now, you are going to create a Python dictionary to test multiple use cases for your agent. Testing your agent is important on its own because it helps ensure that it works correctly in different situations. Automating test cases makes this process easier and helps catch errors before they become a problem. A good test suite checks how the agent handles different inputs, including tricky cases like dividing by zero, working with large numbers, and handling decimals. You can also test how the agent deals with mixed operations, like combining addition and multiplication.


In [ ]:
# Test Cases
test_cases = [
    {
        "query": "Subtract 100, 20, and 10.",
        "expected": {"result": 70},
        "description": "Testing subtraction tool with sequential subtraction."
    },
    {
        "query": "Multiply 2, 3, and 4.",
        "expected": {"result": 24},
        "description": "Testing multiplication tool for a list of numbers."
    },
    {
        "query": "Divide 100 by 5 and then by 2.",
        "expected": {"result": 10.0},
        "description": "Testing division tool with sequential division."
    },
    {
        "query": "Subtract 50 from 20.",
        "expected": {"result": -30},
        "description": "Testing subtraction tool with negative results."
    }

]

This code extracts the actual computation result from the agent's response structure. Unlike a direct tool invocation that returns a simple dictionary, LangGraph agents return a complex structure containing the entire conversation history as a list of messages. To find the computation result, you must locate the specific ToolMessage in this list (identified by its name matching one of the math tools), then parse its content, which contains the actual result as a JSON string. This approach is necessary because the result isn't accessible directly from the response object but is instead nested within the message history, requiring you to navigate through the messages to find and extract the relevant data for comparison with your expected values.


In [ ]:
correct_tasks = []
# Corrected test execution
for index, test in enumerate(test_cases, start=1):
    query = test["query"]
    expected_result = test["expected"]["result"]  # Extract just the value
    
    print(f"\n--- Test Case {index}: {test['description']} ---")
    print(f"Query: {query}")
    
    # Properly format the input
    response = math_agent_new.invoke({"messages": [("human", query)]})
    
    # Find the tool message in the response
    tool_message = None
    for msg in response["messages"]:
        if hasattr(msg, 'name') and msg.name in ['add_numbers', 'new_subtract_numbers', 'multiply_numbers', 'divide_numbers']:
            tool_message = msg
            break
    
    if tool_message:
        # Parse the tool result from its content
        import json
        tool_result = json.loads(tool_message.content)["result"]
        print(f"Tool Result: {tool_result}")
        print(f"Expected Result: {expected_result}")
        
        if tool_result == expected_result:
            print(f"✅ Test Passed: {test['description']}")
            correct_tasks.append(test["description"])
        else:
            print(f"❌ Test Failed: {test['description']}")
    else:
        print("❌ No tool was called by the agent")

print("\nCorrectly passed tests:", correct_tasks)

The current functions would benefit from enhanced error handling and input validation. The add_numbers, subtract_numbers, multiply_numbers, and divide_numbers functions should be updated to handle decimal numbers using float conversion, validate inputs more strictly, and provide clear error messages for edge cases. For example, divide_numbers should explicitly check for division by zero, and all functions should gracefully handle non-numeric inputs like "two" or "hundred". The test cases should be expanded beyond basic operations to include edge cases like divided by zero, empty inputs, and mixed numeric/text inputs (e.g., "divide one hundred by 5"). Also consider adding tests for decimal numbers (e.g., "multiply 3.5 by 2") and sequential operations (e.g., "multiply 10 by 2 then add 5"). This comprehensive testing approach ensures the agent can handle a wide range of real-world mathematical queries.


## **Exploring LangChain's built-in tools**

While creating custom tools is powerful, LangChain provides a rich ecosystem of **pre-built tools** that solve common tasks out of the box. These tools abstract away complex implementation details (API calls, input parsing, error handling) and let you focus on building robust agents quickly.


---

#### **Why use built-in tools?**
- **Reliability**: Tested and maintained by the LangChain community.
- **Time-saving**: No need to reinvent the wheel for common tasks.
- **Integration**: Designed to work seamlessly with LangChain agents.

---


#### **Popular built-in tools**
Here are some widely used tools from `langchain_community.tools`:

| Tool Name               | Description                                                                 |
|-------------------------|-----------------------------------------------------------------------------|
| `WikipediaQueryRun`     | Search Wikipedia for factual information.                                   |
| `GoogleSearchRun`       | Perform web searches using Google’s API.                                    |
| `PythonREPLTool`        | Execute Python code in a safe environment.                                  |
| `OpenWeatherMapQueryRun`| Fetch real-time weather data.                                               |
| `YouTubeSearchTool`     | Search for YouTube videos.                                                  |

---



#### **Example: Using the Wikipedia tool**
Let’s enhance the math agent with Wikipedia access to answer questions requiring factual context.


Now, you'll start by creating a Wikipedia search tool using `@tool` operator. This tool will allow the agent to fetch factual information from Wikipedia when needed.


In [ ]:
from langchain_community.utilities import WikipediaAPIWrapper

# Create a Wikipedia tool using the @tool decorator
@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia for factual information about a topic.
    
    Parameters:
    - query (str): The topic or question to search for on Wikipedia
    
    Returns:
    - str: A summary of relevant information from Wikipedia
    """
    wiki = WikipediaAPIWrapper()
    return wiki.run(query)

In [ ]:
search_wikipedia.invoke("What is tool calling?")

Now, you will **create a list of available tools** (both custom math tools and a using wikipedia tool `search_wikipedia`) and then **initialize an agent** that can use these tools to solve problems. This agent will combine:  
- Custom math tools (`add_numbers`, `new_subtract_numbers`, etc.) for arithmetic operations.  
- A built-in tool (`wiki_tool`, e.g., for Wikipedia searches) for additional functionality.  

By combining these tools, the agent can handle **both mathematical calculations** (e.g., addition, subtraction) and **informational queries** (e.g., fetching facts from Wikipedia), depending on the user’s request.


In [ ]:
# Update your tools list to include the Wikipedia tool
tools_updated = [add_numbers, new_subtract_numbers, multiply_numbers, divide_numbers, search_wikipedia]

# Create the agent with all tools including Wikipedia
math_agent_updated = create_react_agent(
    model=llm_ai,
    tools=tools_updated,
    prompt="You are a helpful assistant that can perform various mathematical operations and look up information. Use the tools precisely and explain your reasoning clearly."
)

Now, you will **ask the agent a multi-step question** that requires:  
1. **Online searching** (using `search_wikipedia` or another built-in tool) to fetch real-world data.  
2. **Mathematical computation** (using `multiply_numbers`) to process the retrieved data.  




In [ ]:
query = "What is the population of Canada? Multiply it by 0.75"

response = math_agent_updated.invoke({"messages": [("human", query)]})

print("\nMessage sequence:")
for i, msg in enumerate(response["messages"]):
    print(f"\n--- Message {i+1} ---")
    print(f"Type: {type(msg).__name__}")
    if hasattr(msg, 'content'):
        print(f"Content: {msg.content}")
    if hasattr(msg, 'name'):
        print(f"Name: {msg.name}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"Tool calls: {msg.tool_calls}")

**How it works**:
1. The agent first uses `search_wikipedia` to find Canada's population.
2. Extracts the numeric value from Wikipedia’s response.
3. Uses `multiply_numbers` to calculate 75% of the population.
4. Returns the final result with context.


For a full list of available tools, see the [LangChain Tools Documentation](https://python.langchain.com/docs/integrations/tools/).


## **Exercise: Create a power tool to calculate exponents**

#### **Objective**
In this exercise, you will create a custom tool that calculates the power of a number (e.g., \( x^y \)). You will then integrate this tool into an agent and test its functionality.

---

#### **Step 1: Create the power tool**

1. **Define the Tool Function**:
   - Create a Python function named `calculate_power` that takes a string as input. The string will contain two numbers: the base (\( x \)) and the exponent (\( y \)).
   - The function should extract the numbers, calculate \( x^y \), and return the result as a dictionary with the key `"result"`.


In [ ]:
#TODO


<details>
    <summary>Click here for Solution</summary>

```python
def calculate_power(input_text: str) -> dict:
    """
    Calculates the power of a number (x^y).

    Parameters:
    - input_text (str): A string like "2, 3", "2 3", "5^2", or "2 to the power of 3".

    Returns:
    - dict: {"result": <calculated value>} or an error message.
    """
    # Try to extract expressions like "5^2"
    match = re.search(r"(\d+(?:\.\d+)?)\s*\^+\s*(\d+(?:\.\d+)?)", input_text)
    if match:
        base = float(match.group(1))
        exponent = float(match.group(2))
        return {"result": base ** exponent}

    # Try to extract expressions like "2 to the power of 3"
    match = re.search(r"(\d+(?:\.\d+)?)\s*(?:to\s+the\s+power\s+of)\s*(\d+(?:\.\d+)?)", input_text, re.IGNORECASE)
    if match:
        base = float(match.group(1))
        exponent = float(match.group(2))
        return {"result": base ** exponent}

    # Fallback: assume two numbers separated by space or comma
    try:
        numbers = [float(num) for num in input_text.replace(",", " ").split()]
        if len(numbers) != 2:
            return {"result": "Invalid input. Please provide exactly two numbers."}
        base, exponent = numbers
        return {"result": base ** exponent}
    except ValueError:
        return {"result": "Invalid input format. Provide input like '2 3', '2^3', or '2 to the power of 3'."}


```

</details>


2. **Create the tool object**:
   - Use the `Tool` class from LangChain to create a tool object for the `calculate_power` function.
   - Provide a name, description, and the function to the tool.


In [ ]:
#TODO


<details>
    <summary>Click here for Solution</summary>

```python
power_tool = Tool(
   name="PowerTool",
   func=calculate_power,
   description="Calculates the power of a number (x^y). Input should be two numbers: base and exponent."
)
```

</details>


#### **Step 2: Create an agent with the power tool**

1. **Set up the agent**:
   - Use the `initialize_agent` function from LangChain to create an agent.
   - Include the `power_tool` in the list of tools provided to the agent.
   - Specify the agent type (e.g., `zero-shot-react-description`).


In [ ]:
#TODO


<details>
    <summary>Click here for Solution</summary>

```python
# List of tools for the agent
tools = [power_tool]

# Create the agent
agent = initialize_agent(
   tools,
   llm,
   agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
   verbose=True,
    handle_parsing_errors=True
)
```

</details>


#### **Step 3: Test the agent**

1. **Test the Agent Using the `run` Function**:
   - Use the `run` function of the agent to test its ability to calculate powers.
   - Pass natural language queries to the agent and observe its responses.


In [ ]:
#TODO


<details>
    <summary>Click here for Solution</summary>

```python
agent.run("Calculate 5 to the power of 2.")
```

</details>


## Authors


[Joseph Santarcangelo](https://author.skills.network/instructors/joseph_santarcangelo)


[Kunal Makwana](https://author.skills.network/instructors/kunal_makwana)


Copyright © IBM Corporation. All rights reserved.
